# Session 2: when two rules disagree, the stricter one wins
**A. Real-world problem.** A bracket sticks out from a wall and carries a load at its tip. It must not break (strength) and it must not sag
more than a given amount (stiffness). Each requirement gives its own minimum thickness, and the plate that is sold comes in fixed thicknesses.
Which rule decides, does it change when the bracket gets longer, and is a steel or an aluminium bracket lighter?

Educational model: a flat rectangular plate of width b and thickness t, fixed at one end (a cantilever) with a load F at the free end,
elastic material. Not a design code: no stress concentration at the fixing, no buckling, no fatigue.

In [ ]:
import math
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"
MATERIALS = {"steel": {"E": 210000.0, "yield": 235.0, "rho": 7850.0},          # E and yield in MPa, rho in kg/m^3
             "aluminium": {"E": 69000.0, "yield": 240.0, "rho": 2700.0}}
PLATES = [2, 3, 4, 5, 6, 8, 10, 12, 15, 20, 25, 30]                              # thicknesses that are sold (mm)

## Step 1: the two rules
For a load F at distance L from the fixing of a plate of width b and thickness t (second moment of area I = b t^3 / 12):

    strength:   the largest stress is at the fixing, sigma = M c / I = 6 F L / (b t^2).   Require sigma <= sigma_allow = yield / SF.
                t >= t_strength = sqrt(6 F L / (b sigma_allow))
    stiffness:  the tip sinks by delta = F L^3 / (3 E I) = 4 F L^3 / (E b t^3).           Require delta <= delta_max.
                t >= t_stiffness = (4 F L^3 / (E b delta_max))^(1/3)

Units: N, mm, MPa (= N/mm^2). The plate must satisfy both, so the required thickness is the larger of the two, and the rule that gives the larger one
is the governing rule.

In [ ]:
def t_strength(force_n, length_mm, width_mm, allow_mpa):
    if min(force_n, length_mm, width_mm, allow_mpa) <= 0:
        raise ValueError("force, length, width and allowable stress must be positive")
    return math.sqrt(6 * force_n * length_mm / (width_mm * allow_mpa))


def t_stiffness(force_n, length_mm, width_mm, e_mpa, delta_max_mm):
    if min(force_n, length_mm, width_mm, e_mpa, delta_max_mm) <= 0:
        raise ValueError("force, length, width, modulus and allowed deflection must be positive")
    return (4 * force_n * length_mm ** 3 / (e_mpa * width_mm * delta_max_mm)) ** (1 / 3)


def tip_deflection(force_n, length_mm, width_mm, thickness_mm, e_mpa):
    return 4 * force_n * length_mm ** 3 / (e_mpa * width_mm * thickness_mm ** 3)


def max_stress(force_n, length_mm, width_mm, thickness_mm):
    return 6 * force_n * length_mm / (width_mm * thickness_mm ** 2)

## Step 2: the rule engine for one material
Compute both thicknesses, take the larger, then round up to the next plate that is sold. Finally check the plate we really have: its
stress, its deflection and its mass. A rule set like this is a small program that turns requirements into a part.

In [ ]:
def next_plate(t_mm):
    for p in PLATES:
        if p >= t_mm * (1 - 1e-12):
            return p
    raise ValueError("no plate is thick enough: change the width, the length or the requirement")


def design(force_n, length_mm, width_mm, delta_max_mm, material="steel", safety_factor=1.5):
    if material not in MATERIALS:
        raise ValueError(f"unknown material '{material}'")
    if safety_factor < 1:
        raise ValueError("the safety factor must be at least 1")
    m = MATERIALS[material]
    allow = m["yield"] / safety_factor
    ts = t_strength(force_n, length_mm, width_mm, allow)
    tf = t_stiffness(force_n, length_mm, width_mm, m["E"], delta_max_mm)
    t_req = max(ts, tf)
    t = next_plate(t_req)
    return {"material": material, "t_strength": ts, "t_stiffness": tf, "governing": "strength" if ts >= tf else "stiffness",
            "t_required": t_req, "t": t, "stress": max_stress(force_n, length_mm, width_mm, t),
            "delta": tip_deflection(force_n, length_mm, width_mm, t, m["E"]), "sf_actual": m["yield"] / max_stress(force_n, length_mm, width_mm, t),
            "mass": width_mm * t * length_mm * m["rho"] * 1e-9}

## Step 3: the governing rule changes with the length, and the lightest material
Strength needs t proportional to sqrt(L) and stiffness needs t proportional to L, so the stiffness rule wins for long brackets and the strength rule for short ones. They meet at the critical length

    L_c = [ (6 F / (b sigma_allow))^(3/2) E b delta_max / (4 F) ]^(2/3)

Different materials need different thicknesses, so we run the same rules for each material and choose the one with the smallest mass. To check that the deflection formula is right we also
integrate the beam equation E I y'' = M(x) = F (L - x) numerically, twice, along the plate.

In [ ]:
def critical_length(force_n, width_mm, e_mpa, allow_mpa, delta_max_mm):
    return ((6 * force_n / (width_mm * allow_mpa)) ** 1.5 * e_mpa * width_mm * delta_max_mm / (4 * force_n)) ** (2 / 3)


def lightest(force_n, length_mm, width_mm, delta_max_mm, safety_factor=1.5):
    """Run the rules for every material and return (best, all results) by smallest mass."""
    results = {name: design(force_n, length_mm, width_mm, delta_max_mm, name, safety_factor) for name in MATERIALS}
    return min(results.values(), key=lambda r: r["mass"]), results


def tip_deflection_numeric(force_n, length_mm, width_mm, thickness_mm, e_mpa, n=4000):
    """Integrate E I y'' = F (L - x) twice from the fixed end with the trapezoid rule. Returns y(L) in mm."""
    x = np.linspace(0, length_mm, n + 1)
    ei = e_mpa * width_mm * thickness_mm ** 3 / 12
    curvature = force_n * (length_mm - x) / ei
    slope = np.concatenate([[0], np.cumsum((curvature[1:] + curvature[:-1]) / 2 * np.diff(x))])
    return float(np.sum((slope[1:] + slope[:-1]) / 2 * np.diff(x)))

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    for mat in MATERIALS:
        r = design(500, 200, 40, 1.0, mat)
        print(f"{mat}: t_strength {r['t_strength']:.2f}, t_stiffness {r['t_stiffness']:.2f} -> {r['governing']} governs, plate {r['t']} mm, mass {r['mass']:.3f} kg, "
              f"stress {r['stress']:.0f} MPa, deflection {r['delta']:.2f} mm")
    best, _ = lightest(500, 200, 40, 1.0)
    print("lightest:", best["material"])
    print(f"critical length for steel: {critical_length(500, 40, 210000, 235 / 1.5, 1.0):.0f} mm")
    print(f"numeric tip deflection {tip_deflection_numeric(500, 200, 40, 12, 210000):.5f} mm, formula {tip_deflection(500, 200, 40, 12, 210000):.5f} mm")

    ls = np.linspace(40, 500, 93)
    fig, ax = plt.subplots(figsize=(6.4, 4.2))
    st = MATERIALS["steel"]
    ax.plot(ls, [t_strength(500, x, 40, st["yield"] / 1.5) for x in ls], color="tab:blue", label="strength rule")
    ax.plot(ls, [t_stiffness(500, x, 40, st["E"], 1.0) for x in ls], color="tab:red", label="stiffness rule")
    ax.plot(ls, [max(t_strength(500, x, 40, st["yield"] / 1.5), t_stiffness(500, x, 40, st["E"], 1.0)) for x in ls], "k--", lw=1, label="required (the larger)")
    ax.axvline(critical_length(500, 40, st["E"], st["yield"] / 1.5, 1.0), color="0.5", ls=":")
    ax.set_xlabel("bracket length L (mm)")
    ax.set_ylabel("thickness (mm)")
    ax.set_title("Steel bracket, F = 500 N, width 40 mm, at most 1 mm sag")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(OUT / "thickness_vs_length.png", dpi=120)

    fig2, ax2 = plt.subplots(figsize=(6.4, 4.2))
    for name, col in (("steel", "tab:blue"), ("aluminium", "tab:green")):
        masses = []
        for x in ls:
            try:
                masses.append(design(500, x, 40, 1.0, name)["mass"])
            except ValueError:                       # no plate that is sold is thick enough
                masses.append(np.nan)
        ax2.plot(ls, masses, "o-", ms=3, color=col, label=name)
    ax2.set_xlabel("bracket length L (mm)")
    ax2.set_ylabel("mass of the bracket (kg)")
    ax2.set_title("Mass of the standard plate for each material")
    ax2.grid(alpha=0.3)
    ax2.legend(fontsize=8)
    fig2.tight_layout()
    fig2.savefig(OUT / "mass_by_material.png", dpi=120)

## Where this stops
The rules here are written in the code, in a fixed order. A real part has dozens of rules that depend on each other, and they should be data that a program can sort, check and change.
Session 3 stores the rules as text, finds the order in which they must be evaluated, rejects circular rules, and draws the part they generate.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
"""Checkpoint for Session 2. Prints PASS or FAIL."""
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    import math
    checks = []
    steel = L.MATERIALS["steel"]
    allow = steel["yield"] / 1.5
    ts = L.t_strength(500, 200, 40, allow)
    tf = L.t_stiffness(500, 200, 40, steel["E"], 1.0)
    checks.append((f"worked example: strength gives t = 9.78 mm (got {ts:.2f})", abs(ts - 9.78) < 0.01))
    checks.append((f"worked example: stiffness gives t = 12.40 mm (got {tf:.2f})", abs(tf - 12.40) < 0.01))
    d = L.design(500, 200, 40, 1.0, "steel")
    checks.append((f"worked example: stiffness governs and the plate is 15 mm (got {d['governing']}, {d['t']} mm)", d["governing"] == "stiffness" and d["t"] == 15))
    checks.append((f"worked example: mass 0.942 kg (got {d['mass']:.3f})", abs(d["mass"] - 0.942) < 0.001))
    checks.append((f"worked example: stress {d['stress']:.0f} MPa is within {allow:.0f} MPa and the sag {d['delta']:.2f} mm within 1 mm", d["stress"] <= allow and d["delta"] <= 1.0))
    da = L.design(500, 200, 40, 1.0, "aluminium")
    checks.append((f"worked example: aluminium needs 20 mm but weighs only 0.432 kg (got {da['t']} mm, {da['mass']:.3f} kg)", da["t"] == 20 and abs(da["mass"] - 0.432) < 0.001))
    checks.append(("the lightest material for this bracket is aluminium", L.lightest(500, 200, 40, 1.0)[0]["material"] == "aluminium"))
    lc = L.critical_length(500, 40, steel["E"], allow, 1.0)
    lo, hi = 10.0, 1000.0
    for _ in range(100):
        mid = (lo + hi) / 2
        if L.t_strength(500, mid, 40, allow) > L.t_stiffness(500, mid, 40, steel["E"], 1.0):
            lo = mid                      # strength still asks for more: the crossing is at a longer length
        else:
            hi = mid
    checks.append((f"validation: the critical length 124.6 mm agrees with a bisection search (got {lc:.1f} and {hi:.1f})", abs(lc - hi) < 1e-6 * lc))
    checks.append(("below the critical length strength governs, above it stiffness governs",
                   L.design(500, 0.8 * lc, 40, 1.0)["governing"] == "strength" and L.design(500, 1.25 * lc, 40, 1.0)["governing"] == "stiffness"))
    ok = True
    for f, l, b, t, e in ((500, 200, 40, 12, 210000), (1200, 150, 30, 8, 69000), (80, 400, 60, 5, 210000)):
        ok &= abs(L.tip_deflection_numeric(f, l, b, t, e) - L.tip_deflection(f, l, b, t, e)) < 1e-5 * L.tip_deflection(f, l, b, t, e)
    checks.append(("validation: integrating the beam equation twice numerically gives the deflection formula to 0.001 percent for three plates", ok))
    checks.append(("doubling the load multiplies the strength thickness by sqrt(2) and the stiffness thickness by 2^(1/3)",
                   abs(L.t_strength(1000, 200, 40, allow) / L.t_strength(500, 200, 40, allow) - math.sqrt(2)) < 1e-12 and abs(L.t_stiffness(1000, 200, 40, 210000, 1) / L.t_stiffness(500, 200, 40, 210000, 1) - 2 ** (1 / 3)) < 1e-12))
    ok = True
    for f, l, b, dm in ((300, 100, 30, 0.5), (800, 180, 50, 2.0), (1500, 120, 60, 3.0), (200, 300, 40, 1.5)):
        for mat in ("steel", "aluminium"):
            try:
                r = L.design(f, l, b, dm, mat)
            except ValueError:
                continue
            m = L.MATERIALS[mat]
            ok &= r["t"] >= r["t_required"] - 1e-9 and r["t"] in L.PLATES and r["stress"] <= m["yield"] / 1.5 + 1e-9 and r["delta"] <= dm + 1e-9
    checks.append(("for 8 cases, the chosen plate is one that is sold, it is not thinner than required, and its stress and sag are within the limits", ok))
    for bad, label in ((lambda: L.design(500, 200, 40, 1.0, "wood"), "an unknown material"), (lambda: L.design(0, 200, 40, 1.0), "a load of zero"),
                       (lambda: L.design(500, 200, 40, 1.0, safety_factor=0.5), "a safety factor below 1"), (lambda: L.design(5000, 500, 20, 0.5, "aluminium"), "a requirement no plate can meet"),
                       (lambda: L.t_stiffness(500, 200, 40, 210000, 0), "an allowed sag of zero")):
        try:
            bad()
            raised = False
        except ValueError:
            raised = True
        checks.append((f"edge case: {label} gives an error", raised))
    return checks
import types
L = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
ok = True
for label, passed in run_checks(L):
    print(('PASS  ' if passed else 'FAIL  ') + label)
    ok &= bool(passed)
print('PASS' if ok else 'FAIL')